#### **The section covers the RAG implementation using PDF Q&A**

In [1]:
import json
# from llama_index.llms.gemini import Gemini
# from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader

from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
import nest_asyncio
nest_asyncio.apply()

In [69]:
# !pip install nest_asyncio

In [57]:
# !pip install llama-index-llms-google-genai llama-index-embeddings-google-genai

#### **Before calling any model lets check the model availability**

In [6]:
# open the json file and read the API key
with open("../backend/config.json") as f:
    API_KEY=json.load(f)["API_KEY"]

In [7]:
import google.generativeai as genai

genai.configure(api_key=API_KEY)

for model in genai.list_models():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [8]:
# Filter only models that support text generation
for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [9]:
for model in genai.list_models():
    if "embedContent" in model.supported_generation_methods:
        print(f"Name: {model.name}")
        print(f"Display name: {model.display_name}")
        print(f"Description: {model.description}")
        print("---")


Name: models/gemini-embedding-001
Display name: Gemini Embedding 001
Description: Obtain a distributed representation of a text.
---
Name: models/gemini-embedding-2-preview
Display name: Gemini Embedding 2 Preview
Description: Obtain a distributed representation of multimodal content.
---
Name: models/gemini-embedding-2
Display name: Gemini Embedding 2
Description: Obtain a distributed representation of multimodal content.
---


In [10]:
# 1. Setup Gemini
Settings.llm = GoogleGenAI(model="models/gemini-2.5-flash", api_key=API_KEY)

Settings.embed_model = GoogleGenAIEmbedding(model="models/gemini-embedding-001", api_key=API_KEY)


In [12]:
# 2. Load PDFs
documents = SimpleDirectoryReader("../pdfs").load_data()

In [13]:
documents

[Document(id_='958ea7ce-99c1-4b32-9ee6-e664487956d0', embedding=None, metadata={'page_label': '1', 'file_name': 'monopoly_instructions.pdf', 'file_path': 'd:\\PDF-QnA-RAG\\notebooks\\..\\pdfs\\monopoly_instructions.pdf', 'file_type': 'application/pdf', 'file_size': 625715, 'creation_date': '2026-03-14', 'last_modified_date': '2026-03-14'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='MONOPOLY \nProperty Trading Game from Parker Brothers" \nAGES 8+ \n2 to 8 Players \nContents: Gameboard, 3 dice, tokens, 32 houses, I2 hotels, Chance \nand Community Chest cards, Title Deed cards, play money and a Banker\'s tray. \nNow there\'s a faster way

In [14]:
# 3. Create Index (Vector Store)
index = VectorStoreIndex.from_documents(documents)

2026-06-14 17:16:15,009 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"


In [35]:
# 4. Query
# query_engine = index.as_query_engine()
query_engine = index.as_chat_engine()
# response = query_engine.query("Summarize the key findings in the PDF.")
response = query_engine.chat("Summarize the key findings in the PDF.")
print(response)

2026-06-14 18:40:48,818 - INFO - Condensed question: Summarize the key findings in the PDF.


2026-06-14 18:40:49,243 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
2026-06-14 18:40:49,357 - INFO - AFC is enabled with max remote calls: 10.
2026-06-14 18:40:58,131 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


The provided documents offer a glimpse into the rules and components of the MONOPOLY Property Trading Game from Parker Brothers.

Here are the key findings:

1.  **Game Overview:**
    *   **Title:** MONOPOLY Property Trading Game.
    *   **Publisher:** Parker Brothers.
    *   **Player Count:** Designed for 2 to 8 players.
    *   **Age Recommendation:** Suitable for ages 8 and up.
    *   **Contents:** The game includes a gameboard, 3 dice (implying two white dice and one Speed Die), tokens, 32 houses, 12 hotels, Chance and Community Chest cards, Title Deed cards, play money, and a Banker's tray.

2.  **Gameplay Options:**
    *   Players can choose to play by the **classic rules** for buying, renting, and selling properties.
    *   Alternatively, they can use the **Speed Die** for a faster game experience. The classic rules are recommended for new players, while experienced players can jump straight to the Speed Die rules.

3.  **Speed Die Rules (for faster play):**
    *   **Star

In [36]:
# ADD THESE DEBUG LINES
print("Type:", type(response))
print("Dir:", dir(response))
print("source_nodes:", response.source_nodes)
print("sources:", response.sources if hasattr(response, 'sources') else "NO SOURCES ATTR")

Type: <class 'llama_index.core.chat_engine.types.AgentChatResponse'>
Dir: ['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__post_init__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'async_response_gen', 'is_dummy_stream', 'metadata', 'response', 'response_gen', 'set_source_nodes', 'source_nodes', 'sources']
source_nodes: [NodeWithScore(node=TextNode(id_='95f22466-67b6-47f6-a91f-eda8c4287a65', embedding=None, metadata={'page_label': '1', 'file_name': 'monopoly_instructions.pdf', 'file_path': 'd:\\PDF-QnA-RAG\\notebooks\\..\\pdfs\\monopoly_instructions.pdf', 'file_type': 'application/pdf', 'file_size': 625715, 'creation_date': '2026-03-14

In [34]:
print(response.source_nodes)

[NodeWithScore(node=TextNode(id_='95f22466-67b6-47f6-a91f-eda8c4287a65', embedding=None, metadata={'page_label': '1', 'file_name': 'monopoly_instructions.pdf', 'file_path': 'd:\\PDF-QnA-RAG\\notebooks\\..\\pdfs\\monopoly_instructions.pdf', 'file_type': 'application/pdf', 'file_size': 625715, 'creation_date': '2026-03-14', 'last_modified_date': '2026-03-14'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='958ea7ce-99c1-4b32-9ee6-e664487956d0', node_type='4', metadata={'page_label': '1', 'file_name': 'monopoly_instructions.pdf', 'file_path': 'd:\\PDF-QnA-RAG\\notebooks\\..\\pdfs\\monopoly_instructions.pdf', 'file_type': 'application/pdf', 'file_size': 625715, 'creation_date': '2026-03-14', 'last_modified_dat

In [33]:
# ADD THESE DEBUG LINES
sources = []

# print("Type:", type(response))
# print("Dir:", dir(response))
# print("source_nodes:", response.source_nodes)
# print("sources:", [node.node for node in response.source_nodes] if hasattr(response, 'source_nodes') else "NO SOURCES ATTR")


if hasattr(response, 'source_nodes') and response.source_nodes:
            for node_with_score in response.source_nodes:
                try:
                    node = node_with_score.node  # Extract the actual node
                    sources.append({
                        "text": node.text[:200] + "...",
                        "page": node.metadata.get("page_label", "N/A"),
                        "file": node.metadata.get("file_name", "Unknown"),
                        "score": round(node_with_score.score, 4) if node_with_score.score else "N/A"
                    })
                except Exception as src_err:
                    print(f"Warning: Could not parse source: {src_err}")
                    continue
print("Extracted sources:", sources)

Extracted sources: [{'text': 'MONOPOLY \nProperty Trading Game from Parker Brothers" \nAGES 8+ \n2 to 8 Players \nContents: Gameboard, 3 dice, tokens, 32 houses, I2 hotels, Chance \nand Community Chest cards, Title Deed cards, play mon...', 'page': '1', 'file': 'monopoly_instructions.pdf', 'score': 0.5565}, {'text': 'instructions and return the card facedown to the bottom of the deck. \nThe "Get Out of Jail Free" card is held until used and then returned to \nthe bottom of the deck. If the player who draws it does n...', 'page': '5', 'file': 'monopoly_instructions.pdf', 'score': 0.5493}]


In [37]:
# TEMP DEBUG - remove after
for s in (response.sources or []):
    print(f"type: {type(s)}")
    print(f"raw_output type: {type(getattr(s, 'raw_output', None))}")
    print(f"raw_output value: {getattr(s, 'raw_output', None)}")
    print("---")


type: <class 'llama_index.core.tools.types.ToolOutput'>
raw_output type: <class 'list'>
raw_output value: [NodeWithScore(node=TextNode(id_='95f22466-67b6-47f6-a91f-eda8c4287a65', embedding=None, metadata={'page_label': '1', 'file_name': 'monopoly_instructions.pdf', 'file_path': 'd:\\PDF-QnA-RAG\\notebooks\\..\\pdfs\\monopoly_instructions.pdf', 'file_type': 'application/pdf', 'file_size': 625715, 'creation_date': '2026-03-14', 'last_modified_date': '2026-03-14'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='958ea7ce-99c1-4b32-9ee6-e664487956d0', node_type='4', metadata={'page_label': '1', 'file_name': 'monopoly_instructions.pdf', 'file_path': 'd:\\PDF-QnA-RAG\\notebooks\\..\\pdfs\\monopoly_instructions.pd

In [38]:
sources = []
if hasattr(response, 'sources') and response.sources:
    for source in response.sources:
        try:
            raw = getattr(source, 'raw_output', None)
            if isinstance(raw, list):
                for node_with_score in raw:
                    if hasattr(node_with_score, 'node'):
                        node = node_with_score.node
                        sources.append({
                            "text": node.text[:200] + "...",
                            "page": node.metadata.get("page_label", "N/A"),
                            "file": node.metadata.get("file_name", "Unknown"),
                            "score": round(node_with_score.score, 4) if node_with_score.score else "N/A"
                        })
        except Exception as src_err:
            print(f"Warning: Could not parse source: {src_err}")
            continue
print("Extracted sources:", sources)

Extracted sources: [{'text': 'MONOPOLY \nProperty Trading Game from Parker Brothers" \nAGES 8+ \n2 to 8 Players \nContents: Gameboard, 3 dice, tokens, 32 houses, I2 hotels, Chance \nand Community Chest cards, Title Deed cards, play mon...', 'page': '1', 'file': 'monopoly_instructions.pdf', 'score': 0.5565}, {'text': 'instructions and return the card facedown to the bottom of the deck. \nThe "Get Out of Jail Free" card is held until used and then returned to \nthe bottom of the deck. If the player who draws it does n...', 'page': '5', 'file': 'monopoly_instructions.pdf', 'score': 0.5493}]


In [71]:
# writing the response to json file
with open("response.json", "w") as f:
    json.dump(str(response), f)